# E3-H2 — Contrastes appariés des ablations locales et voisines

**But** : quantifier l'incertitude des différences de RMSE observées en E3-H, sans réentraînement.

Les comparaisons préspécifiées sont **H3 − H1**, **H3 − H4**, **H3 − H5** et **H1 − persistance**, aux horizons 15, 30, 60 et 120 min.

- Unité de rééchantillonnage : **jour UTC entier**, contenant tous les dispositifs.
- Bootstrap circulaire par blocs de **1, 2, 3 et 5 jours** ; **2 000 réplications** par configuration.
- Différence = moyenne des RMSE par graine du modèle A moins celle du modèle B, sur **les mêmes jours rééchantillonnés**.
- H3 − H5 : les graines sont **appariées**, car H5 est une intervention sur le même modèle entraîné que H3.
- Persistance : un seul prédicteur déterministe, calculé sur les mêmes observations.
- IC percentile à 95 % **descriptifs**, sans correction pour comparaisons multiples ; un IC qui traverse zéro n'établit pas l'équivalence.
- Les **22 jours** constituent une période limitée : cette analyse ne démontre pas la généralisation à d'autres bâtiments ni un effet causal de la sémantique.


In [ ]:
from pathlib import Path
import json, platform
import numpy as np
import pandas as pd

ROOT=Path.cwd().resolve()
SOURCE=ROOT/'outputs'/'E3_H_LOCAL_NEIGHBOR_ABLATION'
if not SOURCE.is_dir() and (ROOT/'e3h_extracted').is_dir():
    SOURCE=ROOT/'e3h_extracted'  # Permet une exécution sur le dossier extrait seul.
OUT=ROOT/'outputs'/'E3_H2_PAIRED_ABLATION_BOOTSTRAP'
PRED=SOURCE/'02_H_common_test_predictions.csv'
METRICS=SOURCE/'01_H_per_seed_metrics.csv'
assert PRED.is_file(),f'Fichier introuvable : {PRED}'
assert METRICS.is_file(),f'Fichier introuvable : {METRICS}'
OUT.mkdir(parents=True,exist_ok=True)

HORIZONS=(15,30,60,120)
SEEDS=(42,123,2026)
LENGTHS=(1,2,3,5)
N_BOOT=2000
RNG_SEED=20261105
PAIRS=[
 ('H3_both','H1_local'),
 ('H3_both','H4_both_shuffled'),
 ('H3_both','H5_both_neighbors_permuted'),
 ('H1_local','Persistence')
]
MODELS={'H1_local','H2_neighbors','H3_both','H4_both_shuffled',
        'H5_both_neighbors_permuted'}
print('Source:',SOURCE)


In [ ]:
df=pd.read_csv(PRED,parse_dates=['timestamp'])
df['timestamp']=pd.to_datetime(df.timestamp,utc=True)
df['uuid']=df.uuid.astype(str)
assert set(df.model)==MODELS,(set(df.model),MODELS)
assert set(df.horizon_min)==set(HORIZONS)
assert set(df.seed)==set(SEEDS)
assert len(df)==60*13643
assert df[['y_true','y_pred','persistence_pred']].notna().all().all()
assert np.isfinite(df[['y_true','y_pred','persistence_pred']].to_numpy()).all()
assert not df.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
print('Loaded rows:',len(df))


In [ ]:
# Audit strict de l'appariement et de la reproductibilité des RMSE.
audits=[]
refs={}
for horizon in HORIZONS:
    s=df[df.horizon_min.eq(horizon)]
    ref=s[(s.model=='H1_local') & (s.seed==SEEDS[0])].sort_values(['uuid','timestamp']).reset_index(drop=True)
    assert len(ref)==13643
    for (model,seed),g in s.groupby(['model','seed']):
        g=g.sort_values(['uuid','timestamp']).reset_index(drop=True)
        assert len(g)==13643
        assert g[['uuid','timestamp']].equals(ref[['uuid','timestamp']]),(horizon,model,seed,'keys')
        assert np.allclose(g.y_true,ref.y_true,rtol=0,atol=1e-6),(horizon,model,seed,'targets')
        assert np.allclose(g.persistence_pred,ref.persistence_pred,rtol=0,atol=1e-6),(horizon,model,seed,'persistence')
        audits.append({'horizon_min':horizon,'model':model,'seed':seed,
                       'keys_ok':True,'targets_ok':True,'persistence_ok':True,'n':len(g)})
    refs[horizon]=ref
assert len(audits)==60
audit=pd.DataFrame(audits)
audit.to_csv(OUT/'01_alignment_audit.csv',index=False)
print('Strict alignment checks passed:',len(audits))

source_metrics=pd.read_csv(METRICS)
rmse_checks=[]
for (h,model,seed),g in df.groupby(['horizon_min','model','seed']):
    rmse=float(np.sqrt(np.mean((g.y_pred.to_numpy()-g.y_true.to_numpy())**2)))
    original=source_metrics[(source_metrics.horizon_min==h)&
                            (source_metrics['mode']==model)&
                            (source_metrics.seed==seed)]
    assert len(original)==1
    delta=abs(rmse-float(original.RMSE.iloc[0]))
    assert delta<1e-6,(h,model,seed,delta)
    rmse_checks.append({'horizon_min':h,'model':model,'seed':seed,
                        'RMSE_recomputed':rmse,'RMSE_source':float(original.RMSE.iloc[0]),
                        'abs_diff':delta})
pd.DataFrame(rmse_checks).to_csv(OUT/'02_RMSE_recomputation_audit.csv',index=False)
print('RMSE checks passed:',len(rmse_checks))


## Bootstrap temporel et analyse de sensibilité

Le point estimé est identique pour toutes les longueurs de blocs. Les intervalles varient selon la dépendance temporelle retenue. Les 3 graines sont **fixées** et moyennées à chaque réplication ; le bootstrap n'est pas une estimation complète de la variabilité entre entraînements.


In [ ]:
def classify(lo,hi):
    return 'A_better' if hi<0 else ('B_better' if lo>0 else 'inconclusive')

result=[]
model_stats=[]
for horizon in HORIZONS:
    ref=refs[horizon]
    dates=ref.timestamp.dt.floor('D')
    unique_days=pd.DatetimeIndex(sorted(dates.unique()))
    D=len(unique_days)
    assert D==22,(horizon,D)
    day_code=unique_days.get_indexer(dates)
    day_n=np.bincount(day_code,minlength=D).astype(float)
    assert day_n.sum()==13643
    squared={}
    for model in sorted(MODELS):
        per_seed=[]
        for seed in SEEDS:
            g=df[(df.horizon_min==horizon)&(df.model==model)&(df.seed==seed)]
            g=g.sort_values(['uuid','timestamp'])
            err=(g.y_pred.to_numpy()-ref.y_true.to_numpy())**2
            per_seed.append(np.bincount(day_code,weights=err,minlength=D))
        squared[model]=np.stack(per_seed)
    persistence_err=(ref.persistence_pred.to_numpy()-ref.y_true.to_numpy())**2
    squared['Persistence']=np.bincount(day_code,weights=persistence_err,minlength=D)[None,:]
    points={model:float(np.sqrt(arr.sum(axis=1)/day_n.sum()).mean())
            for model,arr in squared.items()}
    for model,point in points.items():
        per_seed_rmse=np.sqrt(squared[model].sum(axis=1)/day_n.sum())
        model_stats.append({'horizon_min':horizon,'model':model,
                            'RMSE_mean':point,
                            'RMSE_SD_seeds':float(per_seed_rmse.std(ddof=1))
                                             if len(per_seed_rmse)>1 else np.nan,
                            'n_seeds':len(per_seed_rmse)})
    for L in LENGTHS:
        rng=np.random.default_rng(RNG_SEED+1000*horizon+L)
        starts=rng.integers(0,D,size=(N_BOOT,int(np.ceil(D/L))))
        picks=((starts[:,:,None]+np.arange(L))%D).reshape(N_BOOT,-1)[:,:D]
        counts=np.zeros((N_BOOT,D),dtype=np.int32)
        np.add.at(counts,(np.repeat(np.arange(N_BOOT),D),picks.reshape(-1)),1)
        total_n=counts@day_n
        assert (total_n>0).all()
        boot={model:np.sqrt((arr@counts.T)/total_n[None,:]).mean(axis=0)
              for model,arr in squared.items()}
        for a,b in PAIRS:
            diff=boot[a]-boot[b]
            low,high=np.quantile(diff,[0.025,0.975])
            result.append({'horizon_min':horizon,'block_days':L,'A':a,'B':b,
                           'delta_RMSE':points[a]-points[b],
                           'CI95_low':float(low),'CI95_high':float(high),
                           'conclusion':classify(low,high),
                           'bootstrap_fraction_A_better':float((diff<0).mean()),
                           'n_days':D,'n_nodes':int(day_n.sum()),
                           'bootstrap_replicates':N_BOOT})
    print('Done:',horizon)

contrasts=pd.DataFrame(result)
metrics=pd.DataFrame(model_stats)
assert len(contrasts)==4*4*len(PAIRS)==64
assert len(metrics)==4*6==24
assert contrasts.groupby(['horizon_min','A','B']).delta_RMSE.nunique().eq(1).all()
contrasts.to_csv(OUT/'03_ALL_H2_paired_block_CI.csv',index=False)
metrics.to_csv(OUT/'04_H2_model_RMSE.csv',index=False)

summary=contrasts.pivot(index=['horizon_min','A','B'],
                        columns='block_days',values='conclusion').reset_index()
summary.columns=[str(x) if isinstance(x,int) else x for x in summary.columns]
summary['stable_all_4']=summary[[str(x) for x in LENGTHS]].nunique(axis=1).eq(1)
summary.to_csv(OUT/'00_PRIMARY_H2_sensitivity_summary.csv',index=False)
print(summary.to_string(index=False))
print('Stable conclusions:',int(summary.stable_all_4.sum()),'/',len(summary))


In [ ]:
protocol={
 'experiment':'E3-H2 paired bootstrap for E3-H ablation',
 'input':str(PRED),'source_metrics':str(METRICS),
 'horizons_minutes':list(HORIZONS),'seeds':list(SEEDS),
 'pairs':PAIRS,'block_lengths_days':list(LENGTHS),
 'bootstrap_replicates':N_BOOT,'rng_seed_base':RNG_SEED,
 'unit':'UTC day including all devices; circular moving blocks',
 'metric':'mean across per-seed RMSE; persistence deterministic',
 'CI':'percentile 95%, descriptive, not multiplicity-adjusted',
 'interpretation':'negative delta_RMSE favors A',
 'limitations':['22 days in a single test period','3 fixed seeds',
                'H3 vs H1 differs in effective model capacity',
                'H5 is an inference-time distribution shift',
                'shuffled topology overlaps operational topology',
                'no causal demonstration of semantic-layer benefit'],
 'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}
(OUT/'05_H2_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print('Results saved to:',OUT)


## Fichiers à transmettre

1. `00_PRIMARY_H2_sensitivity_summary.csv` : stabilité des 16 contrastes (4 horizons × 4 paires).
2. `03_ALL_H2_paired_block_CI.csv` : 64 intervalles de confiance (1/2/3/5 jours).
3. `04_H2_model_RMSE.csv` : RMSE par modèle et horizon.
4. `01_alignment_audit.csv`, `02_RMSE_recomputation_audit.csv`, `05_H2_protocol.json` : contrôles et traçabilité.

**À ne pas conclure :** l'absence d'amélioration statistique n'est pas une preuve d'absence d'information voisine ; les contrastes ne prouvent pas l'effet causal de la sémantique Brick/SOSA. La topologie shuffled partage beaucoup d'arêtes avec l'opérationnelle et ne constitue donc qu'un contrôle partiel.
